# Bronze to Silver — v6 scheduled-load preparation

Reads the 12 existing Bronze shortcuts and maintains Silver Delta tables in
`LH_SALES_SILVER_TEST.silver_clean`. This notebook does not build Gold.

**Default: full replacement.** Incremental loading is opt-in per table, and requires
an actual TimestampType modification column that changes on every insert/update.
It uses a bounded timestamp window, overlap, and Delta MERGE. It does not infer CDC
from mirroring. Hard deletes are not captured by timestamp MERGE: opt-in requires
explicit acceptance of retaining them until a periodic full reconciliation.

All selected tables share a batch ID and upper boundary, but Bronze reads are not
an atomic cross-table snapshot. Each table commits independently. Do not overlap
writers; configure pipeline concurrency accordingly. Run Gold only after this whole
notebook succeeds. Partial Silver updates remain after failures.

Attach and pin `LH_SALES_SILVER_TEST` as the default lakehouse. Leave existing
outputs intact. Audit history is appended to `silver_clean._load_audit` and is also
the durable incremental watermark store. An audit-write failure fails the notebook.
Review configuration below before enabling incremental mode.


## 1. Imports and date read/write settings
`functions as F` imports Spark column operations under a short name. `StringType`
identifies text; `StorageLevel` controls caching. `uuid4()` supplies a run identifier.
`datetime.now(timezone.utc)` supplies the processing timestamp.

`configure_spark()` sets both Parquet date/timestamp and INT96 read/write modes
before any source read, and again before each write. Version 5.1 adds the missing
read settings that caused `READ_ANCIENT_DATETIME` in a fresh session.

`CORRECTED` reads/writes without calendar rebasing. This assumes the source values
use the proleptic Gregorian calendar and the outputs have modern readers. When
Parquet writer metadata is unknown, this explicit read mode determines how ancient
values are interpreted. If the source was produced using the legacy Spark 2.x/Hive
hybrid calendar, its read mode must instead be `LEGACY`; the age of a date alone
does not establish which calendar was used. This configuration does not replace
old dates with null or change the cleaning rules.


In [ ]:
import re
import json
from collections import Counter
from datetime import datetime, timezone, timedelta
from time import perf_counter
from uuid import uuid4

from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


def configure_spark():
    spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
    spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
    spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
    spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")


configure_spark()
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Notebook version: 6 | Parquet read/write date modes: CORRECTED")


## Table configuration

Configure each table independently. Full mode requires the entire current source.
Incremental mode requires a confirmed modification timestamp and explicit acceptance
of retained hard deletes. Primary keys and schema validation remain mandatory.
Previews are disabled for scheduled runs. Use TABLES_TO_RUN only for deliberate retries.


In [ ]:
LAKEHOUSE = "LH_SALES_SILVER_TEST"
SOURCE_SCHEMA = "bronze_ref"
TARGET_SCHEMA = "silver_clean"
SHOW_PREVIEW = False
PREVIEW_ROWS = 5
AUDIT_TABLE = f"{LAKEHOUSE}.{TARGET_SCHEMA}._load_audit"
BATCH_ID = str(uuid4())
BATCH_UPPER_BOUND = datetime.now(timezone.utc)
TABLES_TO_RUN = None  # None = all enabled; or ["sales_invoice_line_ext"]

TABLE_KEYS = {
    "customer": ["no"],
    "customer_ext": ["no"],
    "sales_credit_memo_header": ["no"],
    "sales_credit_memo_line": ["document_no", "line_no"],
    "sales_header": ["document_type", "no"],
    "sales_header_ext": ["document_type", "no"],
    "sales_invoice_header": ["no"],
    "sales_invoice_header_ext": ["no"],
    "sales_invoice_line": ["document_no", "line_no"],
    "sales_invoice_line_ext": ["document_no", "line_no"],
    "sales_line": ["document_type", "document_no", "line_no"],
    "sales_line_ext": ["document_type", "document_no", "line_no"],
}


def make_table_config(name, keys):
    text_keys = [key for key in keys if key in {"no", "document_no"}]
    return {
        "name": name,
        "enabled": True,
        "source_table": f"{LAKEHOUSE}.{SOURCE_SCHEMA}.{name}",
        "target_table": f"{LAKEHOUSE}.{TARGET_SCHEMA}.{name}",
        "primary_key": list(keys),
        "trim_columns": list(text_keys),
        "blank_to_null_columns": list(text_keys),
        "required_columns": [],
        "type_conversions": {},
        "drop_columns": [],
        "load_mode": "full",  # "full" or "incremental"
        "watermark_column": None,  # Cleaned column name; must be TimestampType.
        "overlap_minutes": 60,
        "accept_retained_hard_deletes": False,
        "full_snapshot": True,  # This workflow requires the complete current source.
        "allow_empty_source": False,
    }


TABLE_CONFIGS = [make_table_config(name, keys) for name, keys in TABLE_KEYS.items()]
# Example ONLY after confirming the source change-tracking contract:
# for config in TABLE_CONFIGS:
#     if config["name"] == "sales_invoice_line":
#         config.update(load_mode="incremental", watermark_column="systemmodifiedat",
#                       accept_retained_hard_deletes=True)
# No incremental table is enabled by default. Never use posting/invoice date as
# the watermark unless it actually changes for every source insert/update.


## 3. Reusable naming and cleaning functions
- `re.sub()`, `strip()`, and `lower()` produce names such as `document_no`.
- `Counter` detects naming collisions; `toDF(*names)` renames columns in order.
- `require_columns()` checks that rule columns exist before any target write.
- `F.trim()` removes outer ordinary spaces; `when().otherwise()` is SQL-like CASE.
- `withColumn()` returns a DataFrame with an added or replaced column.
- `cast()` applies only explicit conversions, rejecting newly introduced nulls.
  It is not a guarantee against rounding or truncation; choose conversions deliberately.
- `drop()` removes only configured columns. Keys/rule columns cannot be dropped.

By default, only `no` or `document_no` values are trimmed and blank-normalized.
Decimals, dates, other text, and all source rows remain unchanged.

In [ ]:
METADATA_COLUMNS = ["_source_table", "_silver_processed_at", "_silver_run_id"]


def clean_column_name(name):
    cleaned = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    cleaned = re.sub(r"_+", "_", cleaned).strip("_").lower()
    if cleaned and cleaned[0].isdigit():
        cleaned = "col_" + cleaned
    return cleaned


def standardize_column_names(df):
    original_names = df.columns
    cleaned_names = [clean_column_name(name) for name in original_names]

    empty_names = [old for old, new in zip(original_names, cleaned_names) if not new]
    if empty_names:
        raise ValueError(f"Columns would become empty names: {empty_names}")

    collisions = [name for name, count in Counter(cleaned_names).items() if count > 1]
    if collisions:
        raise ValueError(f"Column-name collisions require an explicit mapping: {collisions}")

    reserved = {clean_column_name(name) for name in METADATA_COLUMNS}
    conflicts = sorted(reserved.intersection(cleaned_names))
    if conflicts:
        raise ValueError(f"Source contains reserved lineage names: {conflicts}")

    mapping = list(zip(original_names, cleaned_names))
    return df.toDF(*cleaned_names), mapping


def require_columns(df, columns, purpose):
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Missing columns for {purpose}: {missing}")


def validate_config_columns(df, config):
    keys = config["primary_key"]
    if not keys or len(keys) != len(set(keys)):
        raise ValueError("Configure a nonempty list of distinct primary-key columns.")

    used = (
        keys + config["required_columns"] + config["trim_columns"]
        + config["blank_to_null_columns"] + list(config["type_conversions"])
    )
    require_columns(df, used + config["drop_columns"], "table configuration")

    conflicts = sorted(set(used).intersection(config["drop_columns"]))
    if conflicts:
        raise ValueError(f"Cannot drop columns used by configured rules: {conflicts}")

In [ ]:
def clean_text_columns(df, trim_columns, blank_to_null_columns):
    selected = list(dict.fromkeys(trim_columns + blank_to_null_columns))
    require_columns(df, selected, "text cleaning")
    types = {field.name: field.dataType for field in df.schema.fields}

    nontext = [name for name in selected if not isinstance(types[name], StringType)]
    if nontext:
        raise ValueError(f"Text rules require StringType columns: {nontext}")

    result = df
    for name in selected:
        value = F.col(name)
        if name in trim_columns:
            value = F.trim(value)
        if name in blank_to_null_columns:
            value = F.when(
                F.trim(value) == "", F.lit(None).cast("string")
            ).otherwise(value)
        result = result.withColumn(name, value)

    return result

In [ ]:
def apply_type_conversions(df, conversions):
    require_columns(df, list(conversions), "type conversion")
    result = df

    for name, target_type in conversions.items():
        if not isinstance(target_type, str) or not target_type.strip():
            raise ValueError(f"Provide a Spark type string for {name}.")

        converted = F.col(name).cast(target_type)
        invalid = result.filter(F.col(name).isNotNull() & converted.isNull())
        if invalid.limit(1).count() > 0:
            raise ValueError(f"Conversion to {target_type} would lose values in {name}.")

        result = result.withColumn(name, converted)

    return result


def drop_configured_columns(df, columns):
    require_columns(df, columns, "column removal")
    return df.drop(*columns) if columns else df

## Key validation

Check null/blank keys and duplicate key groups before and after cleaning.
Existence checks stop at one offending result instead of counting every failure.
Duplicate detection still requires grouping; trimming can create collisions, so
both validation stages remain. No source records are silently discarded.


In [ ]:
def validate_primary_key(df, key_columns, label):
    if not key_columns or len(key_columns) != len(set(key_columns)):
        raise ValueError("Supply distinct primary-key columns.")
    require_columns(df, key_columns, "primary key")

    invalid = F.lit(False)
    for name in key_columns:
        invalid = (
            invalid | F.col(name).isNull()
            | (F.trim(F.col(name).cast("string")) == "")
        )

    missing_rows = df.filter(invalid).limit(1).count()
    duplicate_groups = (
        df.groupBy(*key_columns).count()
        .filter(F.col("count") > 1).limit(1).count()
    )

    metrics = {
        "has_missing_keys": bool(missing_rows),
        "has_duplicate_keys": bool(duplicate_groups),
    }
    print(f"{label}: {metrics}")
    if missing_rows or duplicate_groups:
        raise ValueError(f"{label} key validation failed. Inspect records before proceeding.")
    return metrics

## Validate configuration

Validate identifiers, keys, load modes, and incremental prerequisites before writes.
The source remains a full mirrored table; incremental filtering happens here.


In [ ]:
def select_tables(configs, requested=None):
    if TARGET_SCHEMA != "silver_clean":
        raise ValueError("This clean-restart notebook writes only to silver_clean.")
    if SOURCE_SCHEMA == TARGET_SCHEMA:
        raise ValueError("Source and destination schemas must differ.")
    if not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x)
               for x in [LAKEHOUSE, SOURCE_SCHEMA, TARGET_SCHEMA]):
        raise ValueError("Use simple lakehouse and schema identifiers.")
    names = [c["name"] for c in configs]
    if not names or len(names) != len(set(names)):
        raise ValueError("Supply distinct table names.")
    for c in configs:
        if type(c["enabled"]) is not bool:
            raise ValueError("enabled must be a boolean.")
        if not c["enabled"]:
            continue
        name = c["name"]
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name):
            raise ValueError(f"Invalid table name: {name}")
        if c["source_table"] != f"{LAKEHOUSE}.{SOURCE_SCHEMA}.{name}":
            raise ValueError(f"Unexpected source: {c['source_table']}")
        if c["target_table"] != f"{LAKEHOUSE}.{TARGET_SCHEMA}.{name}":
            raise ValueError(f"Unexpected target: {c['target_table']}")
        if c["load_mode"] not in {"full", "incremental"}:
            raise ValueError(f"{name}: unsupported load mode")
        if c["load_mode"] == "incremental":
            if not c["watermark_column"] or c["watermark_column"] in METADATA_COLUMNS:
                raise ValueError(f"{name}: configure a source modification timestamp")
            if type(c["overlap_minutes"]) is not int or c["overlap_minutes"] < 0:
                raise ValueError(f"{name}: overlap_minutes must be a nonnegative integer")
            if c["accept_retained_hard_deletes"] is not True:
                raise ValueError(f"{name}: timestamp loads retain hard deletes; explicitly accept and schedule full reconciliation")
        if c["full_snapshot"] is not True:
            raise ValueError(f"{name}: full replacement requires a complete source table.")
        for key in ["primary_key", "trim_columns", "blank_to_null_columns", "required_columns", "drop_columns"]:
            if not isinstance(c[key], list) or any(not isinstance(x, str) or not x for x in c[key]):
                raise ValueError(f"{name}: {key} must be a list of column names.")
        if not c["primary_key"] or len(set(c["primary_key"])) != len(c["primary_key"]):
            raise ValueError(f"{name}: configure a nonempty distinct primary key.")
        if not isinstance(c["type_conversions"], dict) or type(c["allow_empty_source"]) is not bool:
            raise ValueError(f"{name}: invalid conversion or empty-source settings.")
    enabled = [c for c in configs if c["enabled"]]
    if requested is None:
        selected = enabled
    else:
        if not isinstance(requested, list) or not requested or any(not isinstance(x, str) for x in requested):
            raise ValueError("TABLES_TO_RUN must be None or a nonempty list of table names.")
        unknown = set(requested) - {c["name"] for c in enabled}
        if unknown:
            raise ValueError(f"Unknown or disabled tables requested: {sorted(unknown)}")
        selected = [c for c in enabled if c["name"] in requested]
    if not selected:
        raise ValueError("No enabled tables selected.")
    return selected


def concise_error(error):
    text = str(error).rsplit("Caused by:", 1)[-1]
    return text.split("\n\tat ", 1)[0].strip()[:4000]

## Processing, audit, and watermark rules

Every table attempt appends an audit row with batch/table IDs, status, effective
load mode, counts, UTC boundaries, error, and stage timings. Successful audited
loads provide the next watermark. Failed loads never advance it. If a data commit
succeeds but its audit fails, the next retry safely reprocesses the old window.

The first incremental run bootstraps with a full snapshot. Changing the watermark
column requires another full bootstrap. Incremental reads reject null timestamps
and duplicate keys in the selected changes. They merge all columns by business key;
unchanged target rows retain their previous run ID. Full mode retains count/run-ID
verification. Incremental mode verifies affected keys and their new run IDs.

An empty incremental window is successful; an empty bootstrap/full source remains
blocked by default. Source timestamps must be UTC-compatible, reliable for every
insert/update, and not move backward. Overlap cannot guarantee arbitrary late data.
A shared upper boundary is a processing boundary, not a source snapshot guarantee.


In [ ]:
AUDIT_SCHEMA = (
    "table_name string, target_table string, status string, stage string, "
    "bronze_rows long, silver_rows long, saved_rows long, write_completed boolean, "
    "run_id string, elapsed_seconds double, error string, batch_id string, "
    "effective_mode string, window_lower string, window_upper string, "
    "timings_json string, watermark_column string, completed_at string"
)

def process_table(config):
    started = perf_counter()
    run_id = str(uuid4())
    timings = {}
    stage_started = perf_counter()
    last_stage = "read_source"
    def mark(stage):
        nonlocal stage_started, last_stage
        now = perf_counter()
        timings[last_stage] = timings.get(last_stage, 0.0) + now - stage_started
        last_stage, stage_started = stage, now
        result["stage"] = stage
    result = {
        "table_name": config["name"], "target_table": config["target_table"],
        "batch_id": BATCH_ID, "effective_mode": config["load_mode"],
        "window_lower": None, "window_upper": BATCH_UPPER_BOUND.isoformat(),
        "status": "FAILED", "stage": "read_source", "bronze_rows": None,
        "silver_rows": None, "saved_rows": None, "write_completed": False,
        "run_id": run_id, "elapsed_seconds": 0.0, "error": "",
    }
    bronze, silver = None, None
    try:
        # Protect direct function calls as well as the batch entrypoint.
        select_tables([config])
        configure_spark()
        source = spark.table(config["source_table"])
        incremental = config["load_mode"] == "incremental"
        watermark = config["watermark_column"]
        if incremental:
            from pyspark.sql.types import TimestampType
            named_source, _ = standardize_column_names(source)
            require_columns(named_source, [watermark], "watermark")
            if not isinstance(named_source.schema[watermark].dataType, TimestampType):
                raise ValueError("Watermark must be TimestampType; configure source UTC timestamps explicitly")
            if named_source.filter(F.col(watermark).isNull()).limit(1).count():
                raise ValueError("Null modification timestamps cannot be incrementally loaded")
            prior = (spark.table(AUDIT_TABLE)
                     .filter((F.col("target_table") == config["target_table"])
                             & (F.col("status") == "SUCCESS")
                             & (F.col("watermark_column") == watermark))
                     .agg(F.max("window_upper").alias("upper")).first()["upper"])
            if prior and spark.catalog.tableExists(config["target_table"]):
                lower = datetime.fromisoformat(prior) - timedelta(minutes=config["overlap_minutes"])
                if lower > BATCH_UPPER_BOUND:
                    raise ValueError("Stored watermark is ahead of this run")
                result["window_lower"] = lower.isoformat()
                # Map the cleaned configured name back to the exact source column.
                original = source.columns[named_source.columns.index(watermark)]
                source = source.filter((F.col("`" + original.replace("`", "``") + "`") >= F.lit(lower))
                                       & (F.col("`" + original.replace("`", "``") + "`") < F.lit(BATCH_UPPER_BOUND)))
            else:
                incremental = False
                result["effective_mode"] = "bootstrap_full"
        bronze = source.persist(StorageLevel.MEMORY_AND_DISK)
        result["bronze_rows"] = bronze.count()
        if result["bronze_rows"] == 0 and not incremental and not config["allow_empty_source"]:
            raise ValueError("Source is empty. No target write was attempted.")

        mark("validate_source")
        named, _ = standardize_column_names(bronze)
        validate_config_columns(named, config)
        validate_primary_key(named, config["primary_key"], config["name"] + " source")

        mark("transform")
        cleaned = clean_text_columns(named, config["trim_columns"], config["blank_to_null_columns"])
        cleaned = apply_type_conversions(cleaned, config["type_conversions"])
        cleaned = drop_configured_columns(cleaned, config["drop_columns"])
        silver = (
            cleaned.withColumn("_source_table", F.lit(config["source_table"]))
            .withColumn("_silver_processed_at", F.lit(datetime.now(timezone.utc)))
            .withColumn("_silver_run_id", F.lit(run_id))
            .persist(StorageLevel.MEMORY_AND_DISK)
        )
        result["silver_rows"] = silver.count()
        mark("validate_silver")
        if result["silver_rows"] != result["bronze_rows"]:
            raise ValueError("Unexpected row-count change during cleaning.")
        validate_primary_key(silver, config["primary_key"], config["name"] + " silver")

        if SHOW_PREVIEW:
            mark("preview")
            extra = [c for c in silver.columns if c not in config["primary_key"] + METADATA_COLUMNS][:8]
            display(silver.select(*(config["primary_key"] + extra + METADATA_COLUMNS)).limit(PREVIEW_ROWS))

        target = config["target_table"]
        mark("check_existing_schema")
        mode = "errorifexists"
        if spark.catalog.tableExists(target):
            existing_types = {f.name: f.dataType.simpleString() for f in spark.table(target).schema.fields}
            incoming_types = {f.name: f.dataType.simpleString() for f in silver.schema.fields}
            if existing_types != incoming_types:
                raise ValueError("Target schema differs. Review the schema before replacement.")
            mode = "overwrite"

        mark("write_silver")
        configure_spark()
        if incremental:
            from delta.tables import DeltaTable
            def quoted(name):
                return "`" + name.replace("`", "``") + "`"
            condition = " AND ".join(f"t.{quoted(k)} = s.{quoted(k)}" for k in config["primary_key"])
            (DeltaTable.forName(spark, target).alias("t")
             .merge(silver.alias("s"), condition)
             .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        else:
            silver.write.format("delta").mode(mode).saveAsTable(target)
        result["write_completed"] = True

        mark("verify_saved_data")
        spark.catalog.refreshTable(target)
        saved = spark.table(target)
        if incremental:
            # Verify only affected rows; untouched target rows have older run IDs.
            saved = saved.join(silver.select(*config["primary_key"]), config["primary_key"], "inner")
        metrics = saved.agg(
            F.count(F.lit(1)).alias("rows"),
            F.coalesce(F.sum(F.when(
                F.col("_silver_run_id").isNull() | (F.col("_silver_run_id") != run_id),
                F.lit(1)
            ).otherwise(F.lit(0))), F.lit(0)).alias("wrong_run_rows")
        ).first()
        result["saved_rows"] = metrics["rows"]
        if metrics["rows"] != result["silver_rows"] or metrics["wrong_run_rows"] != 0:
            raise ValueError("Saved row count or run ID differs from this run.")
        result["status"] = "SUCCESS"
        mark("complete")
        print(f"SUCCESS: {target} | {result['saved_rows']:,} rows verified")
    except Exception as error:
        result["error"] = concise_error(error)
        print(f"FAILED: {config['name']} | {result['stage']} | {result['error']}")
    finally:
        for df in [silver, bronze]:
            if df is not None:
                try:
                    df.unpersist()
                except Exception as error:
                    print(f"Cache cleanup warning: {error}")
        timings[last_stage] = timings.get(last_stage, 0.0) + perf_counter() - stage_started
        result["elapsed_seconds"] = round(perf_counter() - started, 3)
        result["timings_json"] = json.dumps({k: round(v, 3) for k, v in timings.items()})
        audit = dict(result, watermark_column=config["watermark_column"],
                     completed_at=datetime.now(timezone.utc).isoformat())
        # Failure here must propagate: do not proceed to Gold without durable audit.
        spark.createDataFrame([audit], AUDIT_SCHEMA).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
    return result

## 7. Run the selected tables and display their results
This is the only execution cell. `CREATE SCHEMA IF NOT EXISTS` creates the output
schema when needed. All destinations use the explicit lakehouse/schema/table name.
The loop calls the same function for each selected table. Failures do not trigger
extra writes; other selected tables are still attempted.

The final exception includes the stage and cause of every failure, plus the exact
`TABLES_TO_RUN` setting for a targeted retry. You do not need a separate diagnostic
cell or another all-table run just to retrieve the error.

In [ ]:
selected_configs = select_tables(TABLE_CONFIGS, TABLES_TO_RUN)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE}.{TARGET_SCHEMA}")
if not spark.catalog.tableExists(AUDIT_TABLE):
    spark.createDataFrame([], AUDIT_SCHEMA).write.format("delta").mode("errorifexists").saveAsTable(AUDIT_TABLE)
print("Batch:", BATCH_ID, "| upper boundary:", BATCH_UPPER_BOUND.isoformat())
print("Destination:", f"{LAKEHOUSE}.{TARGET_SCHEMA}")
print("Selected tables:", [c["name"] for c in selected_configs])

batch_results = []
for config in selected_configs:
    print(f"Processing: {config['name']}")
    batch_results.append(process_table(config))

display(spark.createDataFrame(
    [dict(r, watermark_column=c["watermark_column"], completed_at=None)
     for r, c in zip(batch_results, selected_configs)], AUDIT_SCHEMA))

failed = [r for r in batch_results if r["status"] != "SUCCESS"]
if failed:
    retry_names = [r["table_name"] for r in failed]
    details = "\n\n".join(
        f"{r['table_name']} at {r['stage']}: {r['error']}" for r in failed
    )
    raise RuntimeError(
        f"{len(failed)} table(s) failed.\n\n{details}\n\n"
        f"For a targeted retry, set TABLES_TO_RUN = {retry_names!r} in section 2."
    )

print(f"COMPLETE: all {len(batch_results)} selected tables written and verified.")

## Scheduling and operational use

Run this notebook through a Fabric pipeline with non-overlapping executions.
Leave TABLES_TO_RUN=None for normal scheduled runs. On success, run the separate
Silver-to-Gold notebook, validate Gold, then update the semantic model according
to its storage mode. Start early enough to finish before the report deadline.

The audit table contains one row per table attempt. A complete batch requires all
expected selected tables to have SUCCESS; never infer batch success from one row.
Pipeline notebook success remains the downstream gate. Missing audit rows indicate
an interrupted attempt or audit failure. Retain audit history used by watermarks.

To reconcile hard deletes, temporarily run the relevant table in full mode while
keeping its configured watermark_column, then restore incremental mode. A successful
full reconciliation establishes a new boundary. Incremental changes with source
schema drift fail until a deliberate schema migration is performed.

Test first in the attached test lakehouse: full rerun, initial bootstrap, insert,
update of an older business record, empty window, failure/retry, and hard-delete
reconciliation. This notebook has been validated locally for syntax and control
flow, not executed against Fabric. No exact 12-table source snapshot is promised.
